In [1]:
import pandas as pd 
from pathlib import Path
import subprocess
import time
from tqdm.notebook import tqdm

from utils import * 

In [2]:
save_path = "../results/my_prover_multi_strategy_default_config.csv"
problems_path = "/home/waqee/University_Third_Year/Project/TPTP-v9.0.0/Problems/PUZ/"

proverCommand = "Theorem-Prover"
proverFlags = ["--config", "../configs/multi_strategy_default.json"]
parseOutput = parseProverOutput

In [3]:

problem_files = [str(p.resolve()) for p in Path(problems_path).glob("*.p")]
problem_files = [p for p in problem_files if isClausalFormProblemWithNoAxiomReferences(p) and not withEquality(p)]
problems = [p.removeprefix(problems_path) for p in problem_files]


"Number of problems: ", len(problems)

('Number of problems: ', 44)

In [4]:
df = pd.DataFrame.from_dict(
    {problem: isProblemUnsatisfiable(problems_path+problem) for problem in problems},
    orient="index",
    columns=["unsatisfiable"]
)

for col in ["result", "time"]:
    df[col] = pd.NA

In [ ]:
for problem in tqdm(df.index, desc="Solving Problems", unit="problem"):
    try:
        start_time = time.perf_counter()
        proc = subprocess.run([proverCommand] + proverFlags + [problems_path + problem], capture_output=True, text=True)
        end_time = time.perf_counter()
        
        if proc.returncode != 0:
            df.loc[problem, "result"] = "Exception"
            print(proc.stdout)
            continue
        
        
        proofOutput = parseOutput(proc.stdout.strip())
        df.loc[problem, "result"] = proofOutput
        
        df.loc[problem, "time"] = end_time - start_time if proofOutput != "Timeout" else None
        
    
    except subprocess.TimeoutExpired:
        df.loc[problem, "result"] = "Timeout"
    

Solving Problems:   0%|          | 0/44 [00:00<?, ?problem/s]

In [6]:
incorrect = df[((df["unsatisfiable"] != df["result"]) & (df["result"] != "Timeout") & (df["result"] != "Exception"))]
timeouts = df[df["result"] == "Timeout"]
exceptions = df[df["result"] == "Exception"]
solved = df[((df["unsatisfiable"] == df["result"]))]
avg_solve_time = df[df["result"].isin(["True", "False"])]["time"].mean()

print (f"Incorrect: {len(incorrect)}")
print (f"Timeout: {len(timeouts)}")
print (f"Exception: {len(exceptions)}")
print (f"Solved: {len(solved)}")
print (f"Average Solve Time: {avg_solve_time} seconds")

Incorrect: 0
Timeout: 24
Exception: 0
Solved: 20
Average Solve Time: 1.3043996117990901 seconds


In [7]:
df.to_csv(save_path)

f"Saved results to {save_path}"

'Saved results to ../results/my_prover_multi_strategy_default_config.csv'